# 随机种子与可复现性：让实验可重复
> 难度标签：初级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：13_实用工具 | 源文件：随机种子与可复现性.py | 核心功能：种子设置、环境固定、结果复现

## 概述

可复现性是科学研究的基本要求。ML 实验涉及多处随机性：数据划分、模型初始化、训练过程。需要固定所有随机种子。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import os
import random
import numpy as np

## 数学原理

### 1. 伪随机数生成器（PRNG）

计算机使用确定性算法生成伪随机数。线性同余生成器（LCG）：

$$x_{n+1} = (a \cdot x_n + c) \mod m$$

给定种子 $x_0$，序列完全确定。不同种子产生不同序列。

### 2. 可复现性的数学要求

实验可复现要求：

$$f(x; \theta_{seed=s}) = f(x; \theta_{seed=s}), \quad \forall \text{ runs}$$

即相同种子下，所有随机过程（数据划分、参数初始化、训练顺序）完全一致。

### 3. 各框架的随机性来源

| 框架 | 随机性来源 | 种子设置 |
|------|-----------|----------|
| Python `random` | 随机数模块 | `random.seed(s)` |
| NumPy | 伪随机生成器 | `np.random.seed(s)` |
| PyTorch CPU | CPU 随机数 | `torch.manual_seed(s)` |
| PyTorch GPU | CUDA 随机数 | `torch.cuda.manual_seed_all(s)` |
| sklearn | `random_state` 参数 | `random_state=s` |

### 4. 确定性模式

PyTorch 的确定性模式保证完全可复现：

$$\text{CUBLAS\_WORKSPACE\_CONFIG} = \text{固定}$$

但可能牺牲性能（禁用某些非确定性算法）。

### 5. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `random.seed(42)` | 设置 Python PRNG 种子 $x_0 = 42$ |
| `np.random.seed(42)` | 设置 NumPy PRNG 种子 |
| `torch.manual_seed(42)` | 设置 PyTorch PRNG 种子 |
| `random_state=42` | sklearn 内部 PRNG 种子 |
| `torch.backends.cudnn.deterministic = True` | 强制确定性算法 |

### 1. 各框架随机种子设置

运行 1. 各框架随机种子设置 的代码，观察算法在该环节的行为。

In [ ]:
print("=" * 60)
print("1. 各框架随机种子设置")
print("=" * 60)

SEED = 42

# Python 内置 random
random.seed(SEED)
print(f"random.seed({SEED}) 已设置")

# NumPy
np.random.seed(SEED)
print(f"np.random.seed({SEED}) 已设置")

# PyTorch (CPU)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
# --- 赋值/配置 ---
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    print(f"torch.manual_seed({SEED}) 已设置")
    if torch.cuda.is_available():
        print(f"  CUDA: deterministic=True, benchmark=False")
# --- 继续 ---
except ImportError:
    print("PyTorch 未安装，跳过")

# scikit-learn
print(f"sklearn: 通过 random_state 参数控制 (非全局种子)")

print(f"\n要点: 每个框架有独立的随机状态，需分别设置")

### 2. 验证可复现性 —— NumPy

运行 2. 验证可复现性 —— NumPy 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("2. 验证可复现性 —— NumPy")
print("=" * 60)

# 第一次生成
np.random.seed(SEED)
arr1 = np.random.randn(5)

# 第二次生成
np.random.seed(SEED)
arr2 = np.random.randn(5)

print(f"第1次: {arr1.round(4)}")
print(f"第2次: {arr2.round(4)}")
print(f"结果一致: {np.array_equal(arr1, arr2)}")

### 3. 验证可复现性 —— PyTorch

运行 3. 验证可复现性 —— PyTorch 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("3. 验证可复现性 —— PyTorch")
print("=" * 60)

try:
    import torch
    import torch.nn as nn

    # 第一次
    torch.manual_seed(SEED)
    t1 = torch.randn(5)

    # 第二次
    torch.manual_seed(SEED)
    t2 = torch.randn(5)

    print(f"第1次: {t1.numpy().round(4)}")
    print(f"第2次: {t2.numpy().round(4)}")
    print(f"结果一致: {torch.equal(t1, t2)}")
except ImportError:
    print("PyTorch 未安装，跳过")

### 4. 全局可复现性封装函数

运行 4. 全局可复现性封装函数 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("4. 全局可复现性封装函数")
print("=" * 60)

def set_seed(seed=42):
    """设置所有框架的随机种子，确保实验可复现"""
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
# --- 赋值/配置 ---
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
    except ImportError:
        pass

    try:
        import tensorflow as tf
        tf.random.set_seed(seed)
    except ImportError:
        pass

    try:
        from sklearn.utils import check_random_state
        # sklearn 通过 random_state 参数使用，无需全局设置
    except ImportError:
        pass

    print(f"全局种子已设置: {seed}")

set_seed(SEED)

### 5. 验证封装函数

运行 5. 验证封装函数 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("5. 验证封装函数 —— 多次调用结果一致")
print("=" * 60)

results = []
for run in range(3):
    set_seed(SEED)
    a = np.random.randn(3)
    b = np.random.randint(0, 100, 3)
# --- 继续 ---
    results.append((a.round(4), b))

for i, (a, b) in enumerate(results):
    print(f"  Run {i+1}: arr={a}, int={b}")

all_same_a = all(np.array_equal(r[0], results[0][0]) for r in results)
all_same_b = all(np.array_equal(r[1], results[0][1]) for r in results)
print(f"\n  数组一致: {all_same_a}, 整数一致: {all_same_b}")

### 6. sklearn 模型可复现性

运行 6. sklearn 模型可复现性 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("6. sklearn 模型可复现性")
print("=" * 60)

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.3, random_state=SEED
)

# 两次训练结果一致
accs = []
for run in range(2):
    clf = RandomForestClassifier(n_estimators=50, random_state=SEED)
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
# --- 继续 ---
    accs.append(acc)
    print(f"  Run {run+1}: 准确率={acc:.6f}")

print(f"  两次结果一致: {accs[0] == accs[1]}")

### 7. 常见陷阱

运行 7. 常见陷阱 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("7. 常见陷阱与注意事项")
print("=" * 60)

print("""
陷阱1: 忘记设置某个框架的种子
  错误: 只设了 np.random.seed, 忘了 torch.manual_seed
  后果: PyTorch部分不可复现

陷阱2: 多进程/并行时种子不一致
  原因: 每个worker有独立的随机状态
  解决: 为每个worker设置不同的种子 (base_seed + worker_id)

陷阱3: GPU计算的非确定性
  原因: cuDNN的benchmark和浮点运算顺序
  解决: torch.backends.cudnn.deterministic = True
  代价: 可能降低训练速度

陷阱4: 数据加载顺序
  原因: DataLoader的shuffle和num_workers
  解决: 设置worker_init_fn, 固定shuffle顺序

陷阱5: 版本差异
  不同版本的库可能产生不同结果
  建议: 记录所有依赖版本 (pip freeze)
""")

### 8. 实用技巧

运行 8. 实用技巧 的代码，观察算法在该环节的行为。

In [ ]:
print("=" * 60)
print("8. 实用技巧")
print("=" * 60)

print("1. 项目根目录创建 seed_utils.py, 所有实验导入使用")
print("2. 命令行传入 --seed 参数, 方便对比不同种子")
print("3. 记录种子到日志/配置文件, 方便追溯")
print("4. 关键实验用多个种子运行, 报告均值±标准差")
print("5. 部署时关闭 determinist ic 模式以提升性能")

## 关键代码解释

```python
import random, numpy as np, torch
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.backends.cudnn.deterministic = True
```

## 注意事项

1. 不同硬件/版本可能产生不同结果
2. CUDA 操作有额外随机性
3. 完全可复现可能牺牲性能

## 总结与延伸

以上代码展示了 **随机种子与可复现性** 的完整流程。

**解读要点**：
- 关注工具的 **输入输出格式** 是否正确
- 观察数据加载和预处理的效率
- 检查模型保存/加载后的一致性

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **环境锁定**：requirements.txt、Docker
- **数据版本控制**：DVC
- **确定性训练**：PyTorch 的 determinstic mode
